In [0]:
# paths imports etc
import pandas as pd
attr=r"C:\EveriTools\Workspace\Code\DemandForeCast\Data\raw sample\game_attr (2).csv"
perfp=r"C:\EveriTools\Workspace\Code\DemandForeCast\Data\processed\performance_clean.csv"
mapping=r"C:\EveriTools\Workspace\Code\DemandForeCast\Data\processed\mapping_clean.csv"
rel_map=r"C:\EveriTools\Workspace\Code\DemandForeCast\Data\processed\attr_clean_to_perf_best_match_mapping_all_best_candidates.csv"

In [0]:
# load
attrraw=pd.read_csv(attr)
perf=pd.read_csv(perfp)
map=pd.read_csv(mapping)
rel=pd.read_csv(rel_map)

In [0]:
attrraw

In [0]:
# performance extraction: keep everi/igt and return requested game-level attributes
def _norm_col_name(name: str) -> str:
    return ''.join(ch for ch in str(name).strip().lower() if ch.isalnum())

def _resolve_col(df: pd.DataFrame, candidates: list[str]) -> str:
    norm_to_actual = {_norm_col_name(c): c for c in df.columns}
    for cand in candidates:
        key = _norm_col_name(cand)
        if key in norm_to_actual:
            return norm_to_actual[key]
    raise KeyError(f"None of these columns were found: {candidates}")

def _first_non_null(series: pd.Series):
    non_null = series.dropna()
    return non_null.iloc[0] if not non_null.empty else pd.NA

def extract_perf_game_attributes(
    perf_df: pd.DataFrame,
    allowed_suppliers: tuple[str, ...] = ('everi', 'igt'),
) -> pd.DataFrame:
    supplier_col = _resolve_col(perf_df, ['supplier', 'supplier_cd', 'source_company_cd'])
    game_col = _resolve_col(perf_df, ['game_name', 'game_nm', 'game'])
    parent_own_col = _resolve_col(perf_df, ['parent_own_status', 'parent ownership status', 'parent_owner_status'])
    game_class_col = _resolve_col(perf_df, ['game_classification', 'game classification'])
    release_col = _resolve_col(perf_df, ['game_release_date', 'release_date', 'na_release_date'])

    allowed = {s.strip().lower() for s in allowed_suppliers}
    working = perf_df.copy()
    working[supplier_col] = working[supplier_col].astype(str).str.strip().str.lower()
    working = working[working[supplier_col].isin(allowed)].copy()

    result = (
        working.groupby(game_col, dropna=False, as_index=False)
        .agg(
            **{
                parent_own_col: (parent_own_col, _first_non_null),
                game_class_col: (game_class_col, _first_non_null),
                release_col: (release_col, _first_non_null),
            }
        )
        .rename(
            columns={
                game_col: 'game_name',
                parent_own_col: 'parent_own_status',
                game_class_col: 'game_classification',
                release_col: 'game_release_date',
            }
        )
    )

    return result

perf_game_extract = extract_perf_game_attributes(perf, allowed_suppliers=('everi', 'igt'))
display(perf_game_extract.head(20))
print(f'Rows extracted: {len(perf_game_extract)}')

In [0]:
# clean up
import re
import pandas as pd

attr_clean = attrraw.copy()

THEME_MAPPING = {
    # Ancient Civilizations
    "egyptian": "Ancient Civilizations",
    "incan": "Ancient Civilizations",
    "mayan": "Ancient Civilizations",
    "aztec": "Ancient Civilizations",
    "ancient roman": "Ancient Civilizations",
    "roman": "Ancient Civilizations",
    "greek": "Ancient Civilizations",

    # Fantasy & Mythology
    "fantasy": "Fantasy & Mythology",
    "mythological": "Fantasy & Mythology",
    "genie": "Fantasy & Mythology",
    "fairies": "Fantasy & Mythology",
    "dragons": "Fantasy & Mythology",
    "magic": "Fantasy & Mythology",

    # Nature & Animals
    "animals": "Nature & Animals",
    "animal": "Nature & Animals",
    "underwater": "Nature & Animals",
    "ocean": "Nature & Animals",
    "sea": "Nature & Animals",
    "fishing": "Nature & Animals",
    "jungle": "Nature & Animals",
    "forest": "Nature & Animals",
    "beach": "Nature & Animals",
    "island": "Nature & Animals",
    "fruits": "Nature & Animals",

    # Adventure & Exploration
    "pirates": "Adventure & Exploration",
    "vikings": "Adventure & Exploration",
    "treasure": "Adventure & Exploration",
    "expedition": "Adventure & Exploration",
    "adventure": "Adventure & Exploration",

    # Entertainment & Characters
    "licensed theme": "Entertainment & Characters",
    "licensed": "Entertainment & Characters",
    "character": "Entertainment & Characters",
    "cartoon character": "Entertainment & Characters",
    "movie": "Entertainment & Characters",
    "tv": "Entertainment & Characters",
    "music personality": "Entertainment & Characters",
    "game show": "Entertainment & Characters",
    "carnival": "Entertainment & Characters",
    "carnivale/mardi gras": "Entertainment & Characters",

    # Culture & Regional
    "asian": "Culture & Regional",
    "asian - generic": "Culture & Regional",
    "indian": "Culture & Regional",
    "indian (asian)": "Culture & Regional",
    "latin american": "Culture & Regional",
    "italian": "Culture & Regional",
    "irish": "Culture & Regional",
    "native american": "Culture & Regional",
    "australian": "Culture & Regional",
    "country": "Culture & Regional",
    "western": "Culture & Regional",
    "country / western": "Culture & Regional",

    # Luxury & Classic
    "wealth": "Luxury & Classic",
    "gems": "Luxury & Classic",
    "royalty": "Luxury & Classic",
    "classic": "Luxury & Classic",
    "casino": "Luxury & Classic",

    # Sci-Fi & Mystery
    "space": "Sci-Fi & Mystery",
    "scifi": "Sci-Fi & Mystery",
    "sci-fi": "Sci-Fi & Mystery",
    "mystery": "Sci-Fi & Mystery",
    "horror": "Sci-Fi & Mystery",

    # Seasonal & Miscellaneous
    "winter": "Seasonal & Miscellaneous",
    "heat/fire": "Seasonal & Miscellaneous",
    "fire": "Seasonal & Miscellaneous",
    "art": "Seasonal & Miscellaneous",
    "car": "Seasonal & Miscellaneous",
    "cars": "Seasonal & Miscellaneous",
    "number games": "Seasonal & Miscellaneous",
    "numbers": "Seasonal & Miscellaneous",
    "superstition": "Seasonal & Miscellaneous",

    # Non-theme (Format / Cabinet / Product)
    "video": "Non-theme",
    "stepper": "Non-theme",
    "spinning reel": "Non-theme",
    "multigame": "Non-theme",
    "etg": "Non-theme",
    "bonus bank": "Non-theme",
    "not banked": "Non-theme",
    "flex fusion": "Non-theme",
    "sol sync": "Non-theme",
    "sol sync and flex fusion": "Non-theme",
    "renegade 3600": "Non-theme",
    "pcs premium banked": "Non-theme",
    "empire arena": "Non-theme",
    "barcrest": "Non-theme",
    "dynamic": "Non-theme",
}

def drop_columns_if_exists(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Drop only columns that exist in the dataframe."""
    return df.drop(columns=[c for c in columns if c in df.columns])

def normalize_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Trim + lowercase object columns while preserving nulls."""
    out = df.copy()
    for col in out.select_dtypes(include='object').columns:
        out[col] = out[col].where(
            out[col].isna(),
            out[col].astype(str).str.strip().str.lower()
        )
    return out

def map_game_type_with_theme_dict(
    df: pd.DataFrame,
    mapping: dict[str, str],
    source_col: str = 'game_type',
    target_col: str = 'game_type_theme',
    default_label: str = 'Other',
) -> pd.DataFrame:
    """Map game_type to higher-level themes using exact and contains matching."""
    out = df.copy()
    if source_col not in out.columns:
        return out

    def _map_one(value: object) -> object:
        if pd.isna(value):
            return pd.NA

        text = str(value).strip().lower()
        if not text:
            return pd.NA

        if text in mapping:
            return mapping[text]

        for k, v in mapping.items():
            if k in text:
                return v

        return default_label

    out[target_col] = out[source_col].apply(_map_one)
    return out

def flatten_game_matrix_columns(
    df: pd.DataFrame,
    matrix_cols: list[str],
    drop_original: bool = True,
    invalid_tokens: set[str] | None = None,
    normalize_pattern: str = r"[\[\]{}'\"]",
) -> tuple[pd.DataFrame, list[str], dict[str, list[str]]]:
    """Flatten list-like matrix columns into boolean indicator columns.

    Returns: (updated_df, created_columns, source_to_created_columns).
    """
    out = df.copy()
    invalid = invalid_tokens or {'nan', 'none'}
    created_cols: list[str] = []
    source_map: dict[str, list[str]] = {}

    for gm_col in matrix_cols:
        if gm_col not in out.columns or out[gm_col].dtype != 'object':
            continue

        normalized = (
            out[gm_col]
            .fillna('')
            .astype(str)
            .str.replace(normalize_pattern, '', regex=True)
            .str.replace(r'\s*[,;|]+\s*', '|', regex=True)
            .str.strip('|')
        )

        tokens = normalized.str.split('|')
        unique_tokens = sorted({
            t.strip() for row in tokens for t in row
            if isinstance(t, str) and t.strip() and t.strip() not in invalid
        })

        source_map[gm_col] = []
        for token in unique_tokens:
            safe_token = re.sub(r'[^a-z0-9]+', '_', token).strip('_')
            if not safe_token:
                continue

            new_col = f'{gm_col}_{safe_token}'
            i = 2
            while new_col in out.columns:
                new_col = f'{gm_col}_{safe_token}_{i}'
                i += 1

            out[new_col] = tokens.apply(
                lambda row, tok=token: tok in [x.strip() for x in row if isinstance(x, str)]
            )
            created_cols.append(new_col)
            source_map[gm_col].append(new_col)

        if drop_original:
            out = out.drop(columns=[gm_col])

    return out, created_cols, source_map

def hierarchical_impute(
    df: pd.DataFrame,
    target_cols: list[str],
    hierarchy_cols: list[str],
) -> tuple[pd.DataFrame, dict[str, int]]:
    """Impute target columns using mode within hierarchical groups.

    For each target column, tries grouping by the full hierarchy, then progressively
    less granular groups (left to right) until all possible nulls are filled.
    """
    out = df.copy()
    stats: dict[str, int] = {}

    for target in target_cols:
        if target not in out.columns:
            stats[target] = 0
            continue

        if not out[target].isna().any():
            stats[target] = 0
            continue

        filled_total = 0
        valid_groups = [c for c in hierarchy_cols if c in out.columns and c != target]

        # Most specific -> least specific grouping
        for level in range(len(valid_groups), 0, -1):
            group_cols = valid_groups[:level]
            if not out[target].isna().any():
                break

            non_null = out[out[target].notna()]
            if non_null.empty:
                continue

            mode_lookup = (
                non_null.groupby(group_cols, dropna=False)[target]
                .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else pd.NA)
                .reset_index(name=f'__{target}_mode')
            )

            before = out[target].isna().sum()
            out = out.merge(mode_lookup, on=group_cols, how='left')
            out[target] = out[target].fillna(out[f'__{target}_mode'])
            out = out.drop(columns=[f'__{target}_mode'])
            after = out[target].isna().sum()
            filled_total += int(before - after)

        stats[target] = filled_total

    return out, stats

# 1) Drop unwanted columns
drop_cols = [
    'theme_name',
    #'theme_nk',
    'theme_sk',
    'family_nk',
    'family_name',
    'brand_nk',
    'brand_name',
    'math_model_family',
    'cert',
    'ref',
    'lines',
    'ways',
    'studio_sk',
    'gaming_channel_cd',
    'progressive_tiers',
    'na_release_date',
    'lac_release_date',
    'emea_release_date',
    'apac_release_date',
]
attr_clean = drop_columns_if_exists(attr_clean, drop_cols)

# 2) Normalize string fields
attr_clean = normalize_text_columns(attr_clean)

# 3) Hierarchical imputation (configure these lists per run)
impute_target_cols = ['product_family', 'game_type', 'product_line', 'business_segment', 'rtp_default', 'max_bet', 'min_bet']
impute_hierarchy_cols = ['theme_name_friendly']
attr_clean, impute_stats = hierarchical_impute(
    attr_clean,
    target_cols=impute_target_cols,
    hierarchy_cols=impute_hierarchy_cols,
 )

# 4) Map game_type to theme buckets (after miss-fixing, before flattening)
attr_clean = map_game_type_with_theme_dict(
    attr_clean,
    mapping={k.lower(): v for k, v in THEME_MAPPING.items()},
    source_col='game_type',
    target_col='game_type_theme',
    default_label='Other',
)

# 5) Flatten selected columns
matrix_cols_to_flatten = ['game_matrix', 'game_type_theme']#, 'product_family'
attr_clean, created_game_matrix_cols, matrix_source_map = flatten_game_matrix_columns(
    attr_clean,
    matrix_cols_to_flatten,
    drop_original=True,
    invalid_tokens={'nan', 'none'},
)

print(f'Rows: {len(attr_clean)}')
print(f'Columns: {attr_clean.shape[1]}')
print(f'Flattened boolean columns created: {len(created_game_matrix_cols)}')
print('Imputation fill counts by target column:')
print(impute_stats)

if created_game_matrix_cols:
    display(attr_clean[created_game_matrix_cols].head())

display(attr_clean.head())


In [0]:
from pathlib import Path

desktop = Path.home() / 'Desktop'
desktop.mkdir(parents=True, exist_ok=True)

perf_out =  'perf_game_extract_game_name.csv'
attr_out =  'attr_clean_theme_name_friendly.csv'

perf_game_extract.drop_duplicates().to_csv(perf_out, index=False)
attr_clean.drop_duplicates(subset=['theme_name_friendly']).to_csv(attr_out, index=False)

print(f'Saved: {perf_out}')
print(f'Saved: {attr_out}')
print('Done.')

In [0]:
# Build clustering dataset from rel + attr_clean
import math

def _norm_col_name(name: str) -> str:
    return ''.join(ch for ch in str(name).strip().lower() if ch.isalnum())

def _resolve_col(df: pd.DataFrame, candidates: list[str]) -> str:
    norm_to_actual = {_norm_col_name(c): c for c in df.columns}
    for cand in candidates:
        key = _norm_col_name(cand)
        if key in norm_to_actual:
            return norm_to_actual[key]
    raise KeyError(f"None of these columns were found: {candidates}")

def flatten_categorical_columns(
    df: pd.DataFrame,
    cols: list[str],
    prefix_sep: str = '_',
) -> tuple[pd.DataFrame, list[str]]:
    out = df.copy()
    created_cols: list[str] = []

    for col in cols:
        if col not in out.columns:
            continue

        series = out[col].fillna('').astype(str).str.strip()
        # Keep single-value categories and simple delimited multi-values.
        tokens = series.str.replace(r'\s*[,;|]+\s*', '|', regex=True).str.split('|')
        unique_tokens = sorted({
            t.strip().lower()
            for row in tokens
            for t in row
            if isinstance(t, str) and t.strip() and t.strip().lower() not in {'nan', 'none'}
        })

        for token in unique_tokens:
            safe = ''.join(ch if ch.isalnum() else '_' for ch in token).strip('_')
            if not safe:
                continue
            new_col = f"{col}{prefix_sep}{safe}"
            i = 2
            while new_col in out.columns:
                new_col = f"{col}{prefix_sep}{safe}_{i}"
                i += 1
            out[new_col] = tokens.apply(
                lambda row, tok=token: tok in [x.strip().lower() for x in row if isinstance(x, str)]
            )
            created_cols.append(new_col)

        out = out.drop(columns=[col])

    return out, created_cols

# 1) rel: drop rows with match method == possible_review_fuzzy
rel_match_method_col = _resolve_col(rel, ['match_method', 'match method'])
rel_work = rel.copy()
rel_work = rel_work[
    rel_work[rel_match_method_col].astype(str).str.strip().str.lower() != 'possible_review_fuzzy'
]

# 2) join attr_clean to rel on theme_name_friendly
rel_theme_name_col = _resolve_col(rel_work, ['theme_name_friendly', 'theme_name', 'attr_theme_name_friendly'])
attr_theme_name_col = _resolve_col(attr_clean, ['theme_name_friendly'])

rel_work['join_theme_name'] = rel_work[rel_theme_name_col].astype(str).str.strip().str.lower()
attr_join = attr_clean.copy()
attr_join['join_theme_name'] = attr_join[attr_theme_name_col].astype(str).str.strip().str.lower()

merged_rel_attr = rel_work.merge(
    attr_join,
    on='join_theme_name',
    how='inner',
    suffixes=('_rel', '_attr')
)

# 3) calculate months since released
release_col = _resolve_col(
    merged_rel_attr,
    ['game_release_date', 'release_date', 'candidate_game_release_date']
)
merged_rel_attr['game_release_date_parsed'] = pd.to_datetime(
    merged_rel_attr[release_col], errors='coerce'
)
today_ts = pd.Timestamp.today().normalize()
merged_rel_attr['months_since_released'] = merged_rel_attr['game_release_date_parsed'].apply(
    lambda d: pd.NA if pd.isna(d) else (today_ts.year - d.year) * 12 + (today_ts.month - d.month)
)

# 4) flatten candidat_permium_core and game_classification
premium_col = _resolve_col(
    merged_rel_attr,
    ['candidat_permium_core', 'candidate_premium_core', 'parent_own_status']
)
game_class_col = _resolve_col(
    merged_rel_attr,
    ['game_classification', 'candidate_game_classification']
)

processed_df, created_flat_cols = flatten_categorical_columns(
    merged_rel_attr,
    cols=[premium_col, game_class_col],
)

# 5) holdout most recent 10%, remaining 90% as clustering_df
sort_release_col = 'game_release_date_parsed'
sorted_df = processed_df.sort_values(by=sort_release_col, ascending=False, na_position='last').reset_index(drop=True)
holdout_n = max(1, math.ceil(len(sorted_df) * 0.10)) if len(sorted_df) > 0 else 0

holdout_df = sorted_df.head(holdout_n).copy()
clustering_df = sorted_df.iloc[holdout_n:].copy()

print(f"rel rows after dropping possible_review_fuzzy: {len(rel_work)}")
print(f"rows after attr_clean inner join on theme_name_friendly: {len(merged_rel_attr)}")
print(f"flattened columns created: {len(created_flat_cols)}")
print(f"total rows in processed_df: {len(sorted_df)}")
print(f"holdout rows (10% most recent): {len(holdout_df)}")
print(f"clustering_df rows (90%): {len(clustering_df)}")

display(clustering_df.head(10))

In [0]:
# Agglomerative clustering (full linkage) with tunable feature weights
from sklearn.cluster import AgglomerativeClustering
import numpy as np

def _norm_col_name(name: str) -> str:
    return ''.join(ch for ch in str(name).strip().lower() if ch.isalnum())

def _resolve_existing_cols(df: pd.DataFrame, desired_cols: list[str]) -> tuple[list[str], list[str]]:
    norm_to_actual = {_norm_col_name(c): c for c in df.columns}
    found, missing = [], []
    for c in desired_cols:
        key = _norm_col_name(c)
        if key in norm_to_actual:
            found.append(norm_to_actual[key])
        else:
            missing.append(c)
    return found, missing

# 1) Exact features requested
requested_exact = ['is_wap', 'is_poker', 'is_mlp', 'is_multi_game', 'is_multi_denom','months_since_released']
exact_found, exact_missing = _resolve_existing_cols(clustering_df, requested_exact)

# 2) Prefix feature groups requested
prefix_groups = {
    'parent_own_status_flattened': ['parent_own_status_', 'premium_core_', 'candidate_premium_core_', 'candidat_permium_core_'],
    'game_matrix_features': ['game_matrix_'],
    'game_type_features': ['game_type_', 'game_type_theme_'],
    'game_classification_features': ['game_classification_'],
}

prefix_found = {}
for group_name, prefixes in prefix_groups.items():
    cols = [
        c for c in clustering_df.columns
        if any(c.startswith(p) for p in prefixes)
    ]
    prefix_found[group_name] = sorted(cols)

# Build final feature list
clustering_features = sorted(set(
    exact_found
    + prefix_found['parent_own_status_flattened']
    + prefix_found['game_matrix_features']
    + prefix_found['game_type_features']
    + prefix_found['game_classification_features']
))

if not clustering_features:
    raise ValueError('No clustering features found from requested exact columns/prefix groups.')

# Convert to numeric matrix
X = clustering_df[clustering_features].copy()
for col in X.columns:
    if X[col].dtype == 'bool':
        X[col] = X[col].astype(int)
    else:
        X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

# Log-transform months_since_released: log1p handles 0 and reduces right-skew
if 'months_since_released' in X.columns:
    X['months_since_released'] = np.log1p(X['months_since_released'].clip(lower=0))

# Feature weight dict — edit values to tune individual feature importance
# Set a feature to 0 to exclude it from clustering entirely
feature_weights = {'candidate_premium_core_values': 1.0,
'months_since_released': 1.0,
 'game_classification_class_2': 1.0,
 'game_classification_class_3': 0,
 'game_classification_hhr': 1.0,
 'game_classification_tls': 1.0,
 'game_classification_vgt': 1.0,
 'game_classification_video': 1.0,
 'game_classification_vlt': 1.0,
 'game_matrix_bingo': 1.0,
 'game_matrix_hhr_exacta': 1.0,
 'game_matrix_lottery': 1.0,
 'game_matrix_prize_first_slot': 1.0,
 'game_matrix_reels_first_slot': 1.0,
 'game_type_theme_antasy_ythology': 2.0,
 'game_type_theme_ature_nimals': 2.0,
 'game_type_theme_ci_i_ystery': 2.0,
 'game_type_theme_dventure_xploration': 2.0,
 'game_type_theme_easonal_iscellaneous': 2.0,
 'game_type_theme_ncient_ivilizations': 2.0,
 'game_type_theme_ntertainment_haracters': 2.0,
 'game_type_theme_on_theme': 0.0,
 'game_type_theme_ulture_egional': 2.0,
 'game_type_theme_uxury_lassic': 2.0,
 'is_mlp': 1.0,
 'is_multi_denom': 1.0,
 'is_multi_game': 5.0,
 'is_poker': 1.0,
 'is_wap': 1.0,
 'parent_own_status_core': 0,
 'parent_own_status_premium': 1.0,
 'parent_own_status_premium_ip': 1.0}

# Drop features with weight == 0
zero_weight_features = [f for f, w in feature_weights.items() if w == 0]
active_weights = {f: w for f, w in feature_weights.items() if w != 0}
active_features = [f for f in clustering_features if f in active_weights]
X = X[active_features]

print(f'Features dropped (weight=0): {zero_weight_features}')

# Apply weights
weight_series = pd.Series(active_weights)
X_weighted = X.mul(weight_series, axis=1)

# Full-linkage agglomerative clustering
n_clusters = 8 # tweak as needed
try:
    model = AgglomerativeClustering(n_clusters=n_clusters, linkage='complete', metric='euclidean')
except TypeError:
    # Compatibility for older sklearn versions
    model = AgglomerativeClustering(n_clusters=n_clusters, linkage='complete', affinity='euclidean')

cluster_labels = model.fit_predict(X_weighted)

clustering_scored_df = clustering_df.copy()
clustering_scored_df['agglom_cluster_full'] = cluster_labels

print(f'Exact requested features found: {len(exact_found)} / {len(requested_exact)}')
print(f'Missing exact requested features: {exact_missing}')
print('Prefix-group feature counts:')
for k, v in prefix_found.items():
    print(f'  {k}: {len(v)}')
print(f'Total clustering features used: {len(active_features)} (of {len(clustering_features)} before zero-weight drop)')
print(f'n_clusters: {n_clusters}')

print('Feature weights (feature → weight):')
display(pd.Series(active_weights, name='weight').to_frame())

print('Cluster sizes:')
display(clustering_scored_df['agglom_cluster_full'].value_counts().sort_index())

display(clustering_scored_df[['agglom_cluster_full'] + active_features[:15]].head(10))


In [0]:
X_weighted